# 01 — Görev 1: Veri Ön İşleme (EDA + Outlier + Encoding)

UCI Cleveland Heart Disease veri seti üzerinde:
1. Veri yükleme & hedefin ikili hale getirilmesi
2. Keşifsel veri analizi (sınıf dağılımı, korelasyon, boxplot)
3. Train/Test split (stratified, 80/20)
4. Eksik değer imputation (sadece train medianı ile)
5. IQR ile aykırı değer temizleme (sadece train, sadece sürekli değişkenler)
6. One-Hot Encoding (kategorik değişkenler için)
7. İşlenmiş veriyi `data/processed/` altına kaydetme

**Önemli:** Tüm `fit` işlemleri yalnızca train setinde yapılır, test setine sadece `transform` uygulanır → data leakage yok.


In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\sezgi\Masaüstü\odev-yusuf\heart_disease_project\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd() / "src"))
from src import config as C
from src import utils as U

sns.set_theme(style="whitegrid")
np.random.seed(C.RANDOM_STATE)
print(f"Random state: {C.RANDOM_STATE} | Test size: {C.TEST_SIZE}")

Random state: 42 | Test size: 0.2


## 1. Veriyi yükle

UCI'de `.csv` uzantılı dosya **yoktur** — veri `processed.cleveland.data` adlı,
başlıksız bir dosyada tutulur (içeriği yine virgülle ayrılmıştır). `load_raw`
fonksiyonu veriyi şu sırayla bulmaya çalışır:

1. **Yerel cache** (`data/raw/heart_cleveland.csv`) — daha önce indirildiyse.
2. **Manuel dosya** (`data/raw/processed.cleveland.data`) — sen elle koyduysan.
3. **`ucimlrepo` paketi** — resmi UCI yükleyici, `pip install ucimlrepo` ile gelir. **En güvenilir yöntem.**
4. **Legacy URL** — son çare.

Hiçbiri çalışmazsa fonksiyon, ekrana adım adım manuel indirme talimatı basar.


In [3]:
df = U.load_raw(
    columns=C.COLUMNS,
    save_to=C.RAW_CSV,                            # cache (basliklı CSV dosyasi)
    url=C.UCI_URL,                                # yedek
    manual_data_file=C.MANUAL_DATA_FILE,          # manuel indirme yolu
    dataset_id=C.UCI_DATASET_ID,                  # ucimlrepo id=45
)
print(f"Shape: {df.shape}")
df.head()

Shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [4]:
# Veri tipleri ve eksik değer kontrolü
print(df.dtypes)
print("\nEksik değer (orijinal '?' karakterleri):")
print(df.isna().sum())

age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
num           int64
dtype: object

Eksik değer (orijinal '?' karakterleri):
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: int64


## 2. Hedef değişkeni ikili hale getir
`num > 0` olan tüm vakalar hasta (1), `num == 0` sağlıklı (0).

In [5]:
df[C.TARGET] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])
print(df[C.TARGET].value_counts())
print(f"\nSınıf oranı: {df[C.TARGET].mean():.2%} hasta")

target
0    164
1    139
Name: count, dtype: int64

Sınıf oranı: 45.87% hasta


## 3. EDA — Sınıf dağılımı

In [6]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df[C.TARGET].value_counts().sort_index()
bars = ax.bar(["Sağlıklı (0)", "Hasta (1)"], counts.values,
              color=["#4C9AAA", "#D9534F"], edgecolor="black")
for b, c in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2,
            str(c), ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Gözlem sayısı")
ax.set_title("Sınıf dağılımı")
U.savefig(fig, C.FIG_DIR / "01_class_distribution.png")
print("Kaydedildi → outputs/figures/01_class_distribution.png")

Kaydedildi → outputs/figures/01_class_distribution.png


## 4. EDA — Korelasyon matrisi

In [7]:
# Sayısal kolonlarla korelasyon (kategorikleri de int olarak içerir)
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Korelasyon Matrisi")
U.savefig(fig, C.FIG_DIR / "02_correlation_matrix.png")
print("Kaydedildi → outputs/figures/02_correlation_matrix.png")

Kaydedildi → outputs/figures/02_correlation_matrix.png


## 5. EDA — Aykırı değer öncesi boxplot

In [8]:
fig, axes = plt.subplots(1, len(C.CONTINUOUS), figsize=(3 * len(C.CONTINUOUS), 4))
for ax, col in zip(axes, C.CONTINUOUS):
    sns.boxplot(y=df[col], ax=ax, color="#A8DADC")
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Sürekli özellikler — aykırı değer ÖNCESİ", y=1.02, fontsize=13)
U.savefig(fig, C.FIG_DIR / "03_boxplot_before_outliers.png")
print("Kaydedildi → outputs/figures/03_boxplot_before_outliers.png")

Kaydedildi → outputs/figures/03_boxplot_before_outliers.png


## 6. Train/Test Split

**Sıra önemli:** Split'i imputation ve outlier temizlemeden ÖNCE yapıyoruz.
Bu sayede:
- Imputation için kullanılan medianlar yalnızca train'den hesaplanır.
- IQR sınırları yalnızca train'den hesaplanır.
- Test seti, dış dünyayı simüle eder (kirli + outlier'lı haliyle).


In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[C.TARGET])
y = df[C.TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=C.TEST_SIZE,
    stratify=y,
    random_state=C.RANDOM_STATE,
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train hasta oranı: {y_train.mean():.2%}  |  Test hasta oranı: {y_test.mean():.2%}")

Train: (242, 13)  |  Test: (61, 13)
Train hasta oranı: 45.87%  |  Test hasta oranı: 45.90%


## 7. Eksik değer doldurma (train medianı ile, test'e transform)

In [10]:
# 'ca' ve 'thal' sütunlarında '?' karakterinden gelen NaN'lar var
print("Train eksik:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])
print("\nTest eksik:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

Train eksik:
ca      1
thal    1
dtype: int64

Test eksik:
ca      3
thal    1
dtype: int64


In [11]:
# Sadece train'den hesaplanan median'larla doldur
medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

# Joblib ile median değerleri de saklayalım (Streamlit'te işe yarayacak)
import joblib
joblib.dump(medians.to_dict(), C.MODELS_DIR / "imputation_medians.joblib")
print("Eksik değerler dolduruldu, medianlar models/imputation_medians.joblib altına kaydedildi.")

Eksik değerler dolduruldu, medianlar models/imputation_medians.joblib altına kaydedildi.


## 8. IQR ile aykırı değer temizleme — yalnızca TRAIN
- Yalnızca sürekli değişkenler (`age, trestbps, chol, thalach, oldpeak`) için uygulanır.
- Test setine UYGULANMAZ (gerçek dünya senaryosunu simüle etmek için).

In [12]:
before_n = len(X_train)
combined = X_train.copy()
combined[C.TARGET] = y_train.values
combined_clean, bounds = U.remove_outliers_iqr(combined, C.CONTINUOUS)

X_train = combined_clean.drop(columns=[C.TARGET])
y_train = combined_clean[C.TARGET]

print(f"Train öncesi: {before_n} satır")
print(f"Train sonrası: {len(X_train)} satır  (atılan: {before_n - len(X_train)})")
print("\nIQR sınırları:")
for col, (lo, hi) in bounds.items():
    print(f"  {col:>10s}: [{lo:.2f}, {hi:.2f}]")

Train öncesi: 242 satır
Train sonrası: 227 satır  (atılan: 15)

IQR sınırları:
         age: [28.50, 80.50]
    trestbps: [90.00, 170.00]
        chol: [113.38, 376.38]
     thalach: [87.25, 213.25]
     oldpeak: [-2.40, 4.00]


## 9. EDA — Aykırı değer sonrası boxplot

In [13]:
fig, axes = plt.subplots(1, len(C.CONTINUOUS), figsize=(3 * len(C.CONTINUOUS), 4))
for ax, col in zip(axes, C.CONTINUOUS):
    sns.boxplot(y=X_train[col], ax=ax, color="#9BC4A2")
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Sürekli özellikler — aykırı değer SONRASI (train)", y=1.02, fontsize=13)
U.savefig(fig, C.FIG_DIR / "04_boxplot_after_outliers.png")
print("Kaydedildi → outputs/figures/04_boxplot_after_outliers.png")

Kaydedildi → outputs/figures/04_boxplot_after_outliers.png


## 8. One-Hot Encoding (kategorik değişkenler)

`pd.get_dummies` ile her kategorik değişkeni binary kolonlara aç.
`drop_first=True` ile dummy variable trap'ten kaçınır, model katsayılarının yorumunu kolaylaştırır.
Train/test kolon uyumsuzluğuna karşı `align_dummies` ile doldurma yapılır.

In [14]:
# Kategorikleri açıkça category tipine çevirip get_dummies
X_train_enc = pd.get_dummies(X_train, columns=C.CATEGORICAL, drop_first=True, dtype=int)
X_test_enc = pd.get_dummies(X_test, columns=C.CATEGORICAL, drop_first=True, dtype=int)

# Kolon hizalama (test'te eksik bir kategori varsa o kolonu 0 ile ekle)
X_train_enc, X_test_enc = U.align_dummies(X_train_enc, X_test_enc)

print(f"Encoded train shape: {X_train_enc.shape}")
print(f"Encoded test  shape: {X_test_enc.shape}")
print(f"\nKolonlar ({len(X_train_enc.columns)}): {list(X_train_enc.columns)}")

Encoded train shape: (227, 20)
Encoded test  shape: (61, 20)

Kolonlar (20): ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'sex_1', 'cp_2', 'cp_3', 'cp_4', 'fbs_1', 'restecg_1', 'restecg_2', 'exang_1', 'slope_2', 'slope_3', 'ca_1.0', 'ca_2.0', 'ca_3.0', 'thal_6.0', 'thal_7.0']


## 9. İşlenmiş veriyi kaydet
Sonraki notebook'lar bu dosyalardan başlayacak — `01` notebook'unu tekrar koşturmaya gerek kalmaz.

In [15]:
X_train_enc.to_csv(C.PROC_DIR / "X_train.csv", index=False)
X_test_enc.to_csv(C.PROC_DIR / "X_test.csv", index=False)
y_train.to_csv(C.PROC_DIR / "y_train.csv", index=False)
y_test.to_csv(C.PROC_DIR / "y_test.csv", index=False)

print("Tüm processed dosyalar kaydedildi:")
for f in C.PROC_DIR.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

Tüm processed dosyalar kaydedildi:
  .gitkeep  (0.0 KB)
  X_test.csv  (3.1 KB)
  X_train.csv  (11.2 KB)
  y_test.csv  (0.2 KB)
  y_train.csv  (0.7 KB)



## Üretilen çıktılar:
- `outputs/figures/01_class_distribution.png`
- `outputs/figures/02_correlation_matrix.png`
- `outputs/figures/03_boxplot_before_outliers.png`
- `outputs/figures/04_boxplot_after_outliers.png`
- `data/processed/{X_train, X_test, y_train, y_test}.csv`
- `models/imputation_medians.joblib`

**Sonraki adım:** `02_feature_selection.ipynb`
